In [7]:
# WEBNOVEL TITLE + DESCRIPTION -> GENRE/TAG PREDICTOR

!pip -q install scikit-learn joblib pandas

import pandas as pd
import numpy as np
import re
import joblib

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# csv upload
uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Loaded:", filename)
print("Number of novels:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

# keep only title, description and tags
df = df[["title", "description", "tags"]].copy()
df = df.dropna(subset=["title", "description", "tags"])

df["title"] = df["title"].astype(str)
df["description"] = df["description"].astype(str)
df["tags"] = df["tags"].astype(str)

# remove non-english titles & descriptions
def contains_non_english_characters(text):
    for char in str(text):
        if char.isalpha() and not ("a" <= char.lower() <= "z"):
            return True

    return False

non_english_title = df["title"].apply(
    contains_non_english_characters
)

non_english_description = df["description"].apply(
    contains_non_english_characters
)

# keep only rows where both title and description are english
df = df[
    ~non_english_title
    & ~non_english_description
].copy()

print("\nRows after removing non-English titles/descriptions:", len(df))

print("\nRows after removing missing data:", len(df))

def clean_title(title):
    title = title.lower()

    # Remove URLs
    title = re.sub(r"https?://\S+", " ", title)

    # Replace punctuation with spaces
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # Remove extra spaces
    title = re.sub(r"\s+", " ", title)

    return title.strip()


def clean_description(description):
    description = description.lower()

    # Remove URLs
    description = re.sub(r"https?://\S+", " ", description)

    # Replace punctuation with spaces
    description = re.sub(r"[^a-z0-9\s]", " ", description)

    # Remove extra spaces
    description = re.sub(r"\s+", " ", description)

    return description.strip()


df["clean_title"] = df["title"].apply(clean_title)

df["clean_description"] = df["description"].apply(clean_description)

# combine title x3 and description
df["clean_text"] = (
    df["clean_title"] + " "
    + df["clean_title"] + " "
    + df["clean_title"] + " "
    + df["clean_description"]
)


def parse_tags(tags):
    tag_list = tags.split("|")

    cleaned_tags = []

    for tag in tag_list:
        tag = tag.strip()

        if tag:
            cleaned_tags.append(tag)

    return cleaned_tags


df["tag_list"] = df["tags"].apply(parse_tags)

# remove rows with no tags
df = df[df["tag_list"].apply(len) > 0]

print("Rows with usable tags:", len(df))


# use ALL tags/genres
# no main genre filtering
all_tags = sorted(
    set(
        tag
        for tag_list in df["tag_list"]
        for tag in tag_list
    )
)

print("\n================================================")
print("ALL TAG / GENRE DATASET")
print("================================================")

print("Rows:", len(df))

print("\nNumber of unique tags/genres:", len(all_tags))

print("\nTags/genres:")
print(all_tags)

print("\nTag/genre distribution:")

tag_counts = {}

for tag in all_tags:

    count = df["tag_list"].apply(
        lambda x: tag in x
    ).sum()

    tag_counts[tag] = count

for tag, count in tag_counts.items():
    print(f"{tag:30} {count}")


# convert tags into a multi-label format
mlb = MultiLabelBinarizer(
    classes=all_tags
)

Y = mlb.fit_transform(df["tag_list"])

print("\nNumber of unique tags/genres:", len(mlb.classes_))

print("\nTags/genres:")
print(list(mlb.classes_))



# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    Y,
    test_size=0.20,
    random_state=42
)

print("\nTraining examples:", len(X_train))
print("Testing examples:", len(X_test))

# tf-idf
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF shape:")
print(X_train_tfidf.shape)



# train model
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ),
    n_jobs=-1
)

print("\nTraining model...")

# learns the relationship between tf-idf values and labels
model.fit(X_train_tfidf, y_train)

print("Training complete!")



# evaluate
predictions = model.predict(X_test_tfidf)

print("\n========== MODEL EVALUATION ==========\n")

print(
    classification_report(
        y_test,
        predictions,
        target_names=mlb.classes_,
        zero_division=0
    )
)



# save model
model_data = {
    "model": model,
    "vectorizer": vectorizer,
    "mlb": mlb,
    "genres": all_tags
}

joblib.dump(
    model_data,
    "webnovel_title_description_model.pkl"
)

print("\nModel saved as:")
print("webnovel_title_description_model.pkl")



# predict genres from a title and description
def predict_title(title, description, threshold=0.30):

    cleaned_title = clean_title(title)

    cleaned_description = clean_description(description)

    # combine title and description
    cleaned_text = (
        cleaned_title
        + " "
        + cleaned_description
    )

    # convert title + description to tf-idf
    X = vectorizer.transform([cleaned_text])

    # get probability for every tag
    probabilities = model.predict_proba(X)[0]

    # sort from highest probability to lowest
    results = sorted(
        zip(mlb.classes_, probabilities),
        key=lambda x: x[1],
        reverse=True
    )

    # keep tags above threshold
    predictions = [
        (tag, probability)
        for tag, probability in results
        if probability >= threshold
    ]

    return predictions


# demo
while True:

    print("\n================================================")
    print("DEMO")
    print("================================================")

    title = input("\nEnter a webnovel title (or type 'quit' to stop): ")

    if title.strip().lower() in ["quit", "exit", "q"]:
        print("\nStopping predictor...")
        break

    if title.strip() == "":
        print("\nNo title entered. Stopping predictor...")
        break

    description = input("\nEnter the webnovel description: ")

    results = predict_title(
        title,
        description
    )

    print("\n========== PREDICTIONS ==========\n")

    if len(results) == 0:
        print("No tags passed the threshold.")
        print("Try lowering the threshold.")
    else:
        for tag, probability in results:
            print(f"{tag:30} {probability:.1%}")

    print("\n================================================")
    print("Ready for another novel.")
    print("================================================")

Saving royalroad_novel_metadata.csv to royalroad_novel_metadata.csv
Loaded: royalroad_novel_metadata.csv
Number of novels: 20967

Columns:
['title', 'url', 'authors', 'author_urls', 'tags', 'description', 'overall_score', 'style_score', 'story_score', 'grammar_score', 'character_score', 'total_views', 'average_views', 'followers', 'favorites', 'ratings', 'pages', 'word_count']

First 5 rows:


,title,url,authors,author_urls,tags,description,overall_score,style_score,story_score,grammar_score,character_score,total_views,average_views,followers,favorites,ratings,pages,word_count
0,Reborn as a dungeon core in an apocalyptic world,http://royalroadl.com/fiction/12008/,Myself,http://royalroadl.com/profile/34771,LitRPG | Reincarnation | Anti-Hero Lead | Stra...,"Please delete, thx.",3.76,4.50,5.00,4.50,5.00,11828.0,11828.0,402.0,43.0,119.0,5.0,1483.0
1,Eight God Engine,http://royalroadl.com/fiction/12211/eight-god-...,Vze3vdnp,http://royalroadl.com/profile/60421,Female Lead | Contemporary | Adventure | Fanta...,"Jaq understands running. At sixteen, she escap...",4.54,4.75,4.75,4.88,4.69,15932.0,3983.0,148.0,33.0,49.0,21.0,5851.0
2,Mystic Ink,http://royalroadl.com/fiction/16176/mystic-ink/,vladerag,http://royalroadl.com/profile/76129,Grimdark | Psychological | Female Lead | Drama...,Life is hard for an orphan on the streets of T...,4.34,3.88,3.88,4.62,3.25,288118.0,3893.0,437.0,88.0,145.0,491.0,135248.0
3,"Darkness, Silence And Really Bad Breath",http://royalroadl.com/fiction/6313/,Neliete,http://royalroadl.com/profile/22712,Tragedy | Fantasy | Magic | Supernatural,It was dark; it was very dark for a person tha...,NaN,NaN,NaN,NaN,NaN,4637.0,1159.0,0.0,0.0,0.0,19.0,5293.0
4,(ISSTH CONTEST) The Perfect Dao is Imperfect,http://royalroadl.com/fiction/6727/,KisuDesu,http://royalroadl.com/profile/30262,Martial Arts | Comedy | Action | Adventure | F...,(A tale of Meng Hao) Warning: Tagged 15+ for V...,NaN,NaN,NaN,NaN,NaN,3142.0,3142.0,3.0,5.0,4.0,58.0,16101.0



Rows after removing non-English titles/descriptions: 19664

Rows after removing missing data: 19664
Rows with usable tags: 19664

ALL TAG / GENRE DATASET
Rows: 19664

Number of unique tags/genres: 88

Tags/genres:
['Action', 'Adventure', 'Anti-Hero Lead', 'Anti-Villain Lead', 'Apocalypse', 'Artificial Intelligence', 'Attractive Lead', 'Chivalry', 'Comedy', 'Competing Love Interest', 'Contemporary', 'Cozy', 'Crafting', 'Cultivation', 'Cyberpunk', 'Deck Building', 'Drama', 'Dungeon Core', 'Dungeon Crawler', 'Dystopia', 'Fantasy', 'Female Lead', 'First Contact', 'GameLit', 'Gender Bender', 'Genetically Engineered', 'Grimdark', 'Hard Sci-fi', 'High Fantasy', 'Historical', 'Horror', 'Kingdom Building', 'Lesbian Romance', 'LitRPG', 'Local Protagonist', 'Low Fantasy', 'Magic', 'Magical Girl', 'Magitech', 'Male Gay Romance', 'Male Lead', 'Martial Arts', 'Mecha', 'Modern Knowledge', 'Monster Evolution', 'Multiple Lead Characters', 'Multiple Lovers', 'Mystery', 'Mythos', 'Non-Human Lead', 'Non-

KeyboardInterrupt: Interrupted by user